[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.2_code_copilot/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.2_code_copilot/lab.ipynb)

# Lab 10.2: Code Completion Copilot System Design

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.2_code_copilot/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.2_code_copilot/lab.ipynb)

This lab models the key system design tradeoffs for a code completion copilot:
memory budgeting, speculative decoding throughput, RadixAttention cache savings,
and cost analysis at scale.

In [ ]:
# Install dependencies for plotting and numerical computation
import subprocess
import sys
# Install matplotlib and numpy if not present
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib', 'numpy'])

In [ ]:
import numpy as np  # numerical computation for arrays and math
import matplotlib.pyplot as plt  # plotting library for charts
import matplotlib.patches as mpatches  # custom legend patches

# Configure matplotlib for clean, readable plots
plt.rcParams['figure.figsize'] = (10, 6)  # default figure size
plt.rcParams['font.size'] = 11  # readable font size
plt.rcParams['axes.grid'] = True  # show grid by default
plt.rcParams['grid.alpha'] = 0.3  # subtle grid lines

## Experiment 1: TTFT Budget vs Model Size

The 200ms TTFT budget is the hard constraint. Let's calculate prefill time
for different model sizes at 128K context on H100.

In [ ]:
# === Parameters: Model sizes and their prefill characteristics ===
model_sizes_b = np.array([1, 3, 8, 34, 70])  # model sizes in billions of parameters
# Prefill time at 128K context on single H100 (empirical estimates)
prefill_times_ms = np.array([15, 40, 100, 400, 800])  # milliseconds for full prefill
ttft_budget_ms = 200  # hard TTFT budget in milliseconds

# Calculate which models fit within the TTFT budget
fits_budget = prefill_times_ms <= ttft_budget_ms  # boolean mask for feasible models

# Create bar chart showing prefill time vs budget
fig_c2, ax_c2 = plt.subplots(figsize=(10, 6))  # create figure and axis
colors = ['#dcfce7' if fits else '#ffe4e6' for fits in fits_budget]  # green=feasible, red=infeasible
bars = ax_c2.bar(range(len(model_sizes_b)), prefill_times_ms, color=colors, edgecolor='#000', linewidth=1.2)

# Draw the 200ms TTFT budget line
ax_c2.axhline(y=ttft_budget_ms, color='#991b1b', linestyle='--', linewidth=2, label=f'TTFT Budget ({ttft_budget_ms}ms)')

# Label each bar with its prefill time
for i, (bar, time) in enumerate(zip(bars, prefill_times_ms)):
    label = f'{time}ms\n{"PASS" if fits_budget[i] else "FAIL"}'  # show time and pass/fail
    ax_c2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, label,
            ha='center', fontsize=10, fontweight='bold',
            color='#166534' if fits_budget[i] else '#991b1b')

# Formatting
ax_c2.set_xticks(range(len(model_sizes_b)))  # set x-axis tick positions
ax_c2.set_xticklabels([f'{s}B' for s in model_sizes_b])  # label with model sizes
ax_c2.set_xlabel('Model Size (Parameters)')  # x-axis label
ax_c2.set_ylabel('Prefill Time (ms) at 128K Context')  # y-axis label
ax_c2.set_title('Code Copilot: Model Size vs TTFT Budget (H100, 128K context)')  # chart title
ax_c2.legend(loc='upper left')  # show legend
ax_c2.set_ylim(0, 1000)  # set y-axis range
plt.tight_layout()  # prevent label clipping
plt.show()  # render the chart

# Print conclusion
print(f'\nConclusion: Only models <= 8B fit within {ttft_budget_ms}ms TTFT budget at 128K context.')
print(f'The 8B model at 100ms leaves 100ms headroom for network + queue + transfer.')

## Experiment 2: Speculative Decoding Throughput

Speculative decoding pairs a fast draft model with a larger verifier.
Code has high token predictability (75-85% acceptance rate), making
speculative decoding exceptionally effective.

In [ ]:
# === Parameters: Speculative decoding configuration ===
draft_k = 5  # number of draft tokens generated per step
draft_time_per_token_ms = 0.6  # 1B model generates at 0.6ms/token
verify_time_ms = 5.0  # time for verifier to check K tokens in one pass
baseline_decode_ms = 5.0  # baseline: 8B model without speculation (ms/token)
acceptance_rates = np.linspace(0.3, 0.95, 50)  # sweep acceptance rates

# Calculate effective tokens per speculative step
# With acceptance rate alpha, expected accepted tokens = sum(alpha^i for i in 1..K)
def effective_tokens_per_step(alpha, k):
    """Calculate expected accepted tokens per speculative decoding step."""
    # Geometric series: alpha + alpha^2 + ... + alpha^k = alpha*(1-alpha^k)/(1-alpha)
    if alpha >= 1.0:
        return k  # perfect acceptance
    return alpha * (1 - alpha**k) / (1 - alpha)  # expected accepted tokens

# Calculate speedup for each acceptance rate
effective_toks = np.array([effective_tokens_per_step(a, draft_k) for a in acceptance_rates])
# Time per speculative step = draft time + verify time
step_time_ms = (draft_k * draft_time_per_token_ms) + verify_time_ms  # total time per step
# Effective ms per token with speculation
spec_ms_per_token = step_time_ms / effective_toks  # amortized time per accepted token
# Speedup over baseline
speedup = baseline_decode_ms / spec_ms_per_token  # speedup factor

# Plot speedup vs acceptance rate
fig_c3, (ax1_c3, ax2_c3) = plt.subplots(1, 2, figsize=(14, 5))  # two side-by-side plots

# Left plot: speedup curve
ax1_c3.plot(acceptance_rates * 100, speedup, color='#2563eb', linewidth=2.5)  # main curve
# Highlight the code-specific range (75-85%)
code_mask = (acceptance_rates >= 0.75) & (acceptance_rates <= 0.85)  # code acceptance range
ax1_c3.fill_between(acceptance_rates[code_mask] * 100, speedup[code_mask], alpha=0.3, color='#dcfce7',
                 label='Code range (75-85%)')  # shade the code-specific zone
ax1_c3.axhline(y=1.0, color='gray', linestyle=':', label='No speedup (1x)')  # baseline reference
ax1_c3.set_xlabel('Draft Token Acceptance Rate (%)')  # x-axis
ax1_c3.set_ylabel('Decode Speedup (x)')  # y-axis
ax1_c3.set_title(f'Speculative Decoding Speedup (K={draft_k}, Draft=1B, Verify=8B)')  # title
ax1_c3.legend()  # show legend

# Right plot: time breakdown comparison
categories = ['Baseline\n(8B only)', 'Spec Decode\n(75% accept)', 'Spec Decode\n(85% accept)']
tokens_to_generate = 50  # typical code completion length

# Calculate total decode time for 50 tokens
baseline_total = tokens_to_generate * baseline_decode_ms  # baseline total time
spec_75_ms_per_tok = step_time_ms / effective_tokens_per_step(0.75, draft_k)  # at 75% acceptance
spec_85_ms_per_tok = step_time_ms / effective_tokens_per_step(0.85, draft_k)  # at 85% acceptance
totals = [baseline_total, tokens_to_generate * spec_75_ms_per_tok, tokens_to_generate * spec_85_ms_per_tok]

# Bar chart comparing decode times
bar_colors = ['#ffe4e6', '#fef3c7', '#dcfce7']  # red, amber, green
bars2 = ax2_c3.bar(categories, totals, color=bar_colors, edgecolor='#000', linewidth=1.2)
for bar, total in zip(bars2, totals):  # label each bar with its time
    ax2_c3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{total:.0f}ms', ha='center', fontweight='bold')
ax2_c3.set_ylabel('Total Decode Time for 50 Tokens (ms)')  # y-axis
ax2_c3.set_title('Decode Time: 50-Token Code Completion')  # title

plt.tight_layout()  # prevent overlap
plt.show()  # render

# Print key numbers
print(f'At 80% acceptance: {baseline_decode_ms / (step_time_ms / effective_tokens_per_step(0.80, draft_k)):.1f}x speedup')
print(f'50 tokens at 80% acceptance: {tokens_to_generate * step_time_ms / effective_tokens_per_step(0.80, draft_k):.0f}ms (vs {baseline_total:.0f}ms baseline)')

## Experiment 3: RadixAttention Cache Economics

RadixAttention stores KV cache in a radix tree, enabling prefix sharing.
For code copilots, successive keystrokes share 99.9% of context.

In [ ]:
# === Parameters: Session and caching characteristics ===
cold_start_prefill_ms = 100  # full 128K prefill time in milliseconds
incremental_prefill_ms = 0.1  # time to process 1 new token (cache hit)
keystrokes_per_session = np.arange(1, 201)  # simulate 1-200 keystrokes in a session

# Calculate cumulative TTFT with and without caching
# Without cache: every keystroke triggers full prefill
cumulative_no_cache = cold_start_prefill_ms * keystrokes_per_session  # linear growth
# With RadixAttention: first keystroke is cold, rest are incremental
cumulative_with_cache = cold_start_prefill_ms + (keystrokes_per_session - 1) * incremental_prefill_ms

# Calculate average TTFT across the session
avg_no_cache = cumulative_no_cache / keystrokes_per_session  # constant at 100ms
avg_with_cache = cumulative_with_cache / keystrokes_per_session  # decreasing toward 0.1ms

# Plot the comparison
fig_c4, (ax1_c4, ax2_c4) = plt.subplots(1, 2, figsize=(14, 5))  # two plots

# Left: average TTFT per keystroke across session
ax1_c4.plot(keystrokes_per_session, avg_no_cache, color='#991b1b', linewidth=2, label='Without Cache')  # flat at 100ms
ax1_c4.plot(keystrokes_per_session, avg_with_cache, color='#166534', linewidth=2, label='With RadixAttention')  # rapidly decreasing
ax1_c4.axhline(y=200, color='gray', linestyle=':', alpha=0.5, label='TTFT Budget (200ms)')  # budget line
ax1_c4.set_xlabel('Keystrokes in Session')  # x-axis
ax1_c4.set_ylabel('Average TTFT (ms)')  # y-axis
ax1_c4.set_title('Average TTFT Across Editing Session')  # title
ax1_c4.set_yscale('log')  # log scale to show the dramatic difference
ax1_c4.legend()  # show legend
ax1_c4.set_ylim(0.05, 250)  # y range

# Right: cumulative compute savings
compute_saved_pct = (1 - cumulative_with_cache / cumulative_no_cache) * 100  # percentage saved
ax2_c4.plot(keystrokes_per_session, compute_saved_pct, color='#2563eb', linewidth=2.5)  # savings curve
ax2_c4.fill_between(keystrokes_per_session, compute_saved_pct, alpha=0.2, color='#dbeafe')  # shade area
ax2_c4.set_xlabel('Keystrokes in Session')  # x-axis
ax2_c4.set_ylabel('Compute Savings (%)')  # y-axis
ax2_c4.set_title('RadixAttention Compute Savings vs No-Cache Baseline')  # title
ax2_c4.set_ylim(0, 100)  # percentage range

plt.tight_layout()  # prevent overlap
plt.show()  # render

# Print key takeaways
print(f'After 100 keystrokes: avg TTFT = {avg_with_cache[99]:.1f}ms (vs {avg_no_cache[99]:.0f}ms without cache)')
print(f'Compute savings at 100 keystrokes: {compute_saved_pct[99]:.1f}%')
print(f'Compute savings at 200 keystrokes: {compute_saved_pct[199]:.1f}%')

## Experiment 4: Fleet Cost Model

Model the infrastructure cost for serving 500M completions/day
with disaggregated prefill/decode on H100 TP=2 nodes.

In [ ]:
# === Parameters: Fleet cost model ===
daily_requests = 500_000_000  # total completions per day
cache_hit_rate = 0.95  # RadixAttention cache hit rate
avg_tokens_output = 50  # average output length per completion
h100_cost_per_hour = 3.50  # cloud cost per H100 GPU per hour
gpus_per_node = 2  # TP=2 means 2 GPUs per node

# Throughput per node (with speculative decoding)
completions_per_second_per_node = 400  # 50-token completions per second
completions_per_hour_per_node = completions_per_second_per_node * 3600  # per hour

# Effective compute needed (cache hits are nearly free)
cache_miss_requests = daily_requests * (1 - cache_hit_rate)  # requests needing full compute
cache_hit_requests = daily_requests * cache_hit_rate  # requests served from cache (minimal compute)

# Nodes needed for sustained throughput (24hr average)
# Cache hits still need decode but not prefill, so they're ~5x cheaper
effective_full_requests = cache_miss_requests + cache_hit_requests * 0.2  # cache hits are 5x cheaper
sustained_nodes = effective_full_requests / (completions_per_hour_per_node * 24)  # nodes for sustained
peak_nodes = sustained_nodes * 3  # 3x for peak traffic

# Cost calculation
node_cost_per_hour = h100_cost_per_hour * gpus_per_node  # cost per node per hour
daily_cost = peak_nodes * node_cost_per_hour * 24  # daily infrastructure cost
monthly_cost = daily_cost * 30  # monthly cost
cost_per_completion = monthly_cost / (daily_requests * 30)  # cost per single completion
cost_per_user_month = monthly_cost / 10_000_000  # cost per user per month (10M DAU)

# Print cost summary
print('=== Code Copilot Fleet Cost Model ===')
print(f'Daily requests: {daily_requests:,.0f}')
print(f'Cache hit rate: {cache_hit_rate*100:.0f}%')
print(f'Sustained nodes needed: {sustained_nodes:.0f}')
print(f'Peak nodes (3x): {peak_nodes:.0f}')
print(f'Total GPUs (peak): {peak_nodes * gpus_per_node:.0f} H100s')
print(f'\n--- Monthly Costs ---')
print(f'Monthly infrastructure: ${monthly_cost:,.0f}')
print(f'Cost per completion: ${cost_per_completion:.7f}')
print(f'Cost per user/month (10M DAU): ${cost_per_user_month:.3f}')

# Visualize cost breakdown by component
fig_c5, ax_c5 = plt.subplots(figsize=(8, 5))  # create figure
# Pie chart of where money goes
components = ['Prefill Pool\n(cache misses)', 'Decode Pool\n(all requests)', 'Networking\n(RDMA)', 'Edge/LB']  # cost components
pcts = [0.35, 0.45, 0.12, 0.08]  # estimated cost distribution
pie_colors = ['#fef3c7', '#dcfce7', '#dbeafe', '#f3f4f6']  # pastel colors
wedges, texts, autotexts = ax_c5.pie(pcts, labels=components, autopct='%1.0f%%',
                                   colors=pie_colors, startangle=90,
                                   wedgeprops={'edgecolor': '#000', 'linewidth': 1})  # styled pie
ax_c5.set_title(f'Monthly Cost Distribution (${monthly_cost:,.0f}/month)')  # title with total
plt.tight_layout()  # prevent clipping
plt.show()  # render

# Revenue comparison
print(f'\n--- Revenue Comparison ---')
paying_users = 1_000_000  # 1M paying users (10% of DAU)
revenue_per_user = 10  # $10/month individual plan
monthly_revenue = paying_users * revenue_per_user  # total revenue
print(f'Revenue (1M users at $10/mo): ${monthly_revenue:,.0f}')
print(f'Gross margin: {(1 - monthly_cost/monthly_revenue)*100:.1f}%')